# Experiment 02: Seasonal validation of mlp_4out_mean_bn

Card: [exp02_seasons.md](../../experiments/exp02_seasons.md)

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, "../../..")  # the ashrae package lives in the homework folder
from ashrae.config import FOLDS
from ashrae.data import read_processed, split
from ashrae.features import add_series_mean, build_features
from ashrae.metrics import rmsle
from ashrae.modeling import MeterMLP, make_loaders, predict
from ashrae.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
NOTEBOOK = "exp02_seasons"
if SMOKE:
    EXPERIMENT = "ashrae-validation-seasons-smoke"
    TRAIN = True
    FOLD_NAMES = list(FOLDS)[:2]
    N_TRAIN, N_VAL = 20_000, 5_000
    EPOCHS = 1
    FIGURE = None
else:
    EXPERIMENT = "ashrae-validation-seasons"
    TRAIN = False  # the runs below are from 2026-09-27; True retrains them under a new session
    FOLD_NAMES = list(FOLDS)
    N_TRAIN, N_VAL = 2_000_000, 500_000
    EPOCHS = 10
    FIGURE = "../../figures/exp02_seasons.png"
SESSION = None if TRAIN else "2026-09-27 12:43:17"

experiment = Experiment(EXPERIMENT, NOTEBOOK, SESSION)

## Data

In [ ]:
df = read_processed()
anomaly_share, folds = {}, {}
for fold in FOLD_NAMES:
    train_df, val_df = split(df, FOLDS[fold], N_TRAIN, N_VAL)
    anomaly_share[fold] = val_df.anomaly.mean()
    if TRAIN:
        year_median = train_df.year_built.median()
        # series mean from the fold's training rows; series without training rows get their meter type's mean
        X_train = add_series_mean(build_features(train_df, year_median), train_df, train_df)
        X_val = add_series_mean(build_features(val_df, year_median), val_df, train_df)
        folds[fold] = (*make_loaders(train_df, val_df, X_train, X_val)[:2], val_df.anomaly.to_numpy())
del df
print(", ".join(f"{fold} {share:.1%}" for fold, share in anomaly_share.items()), "of validation rows are anomalies")

## Training

In [ ]:
if TRAIN:
    n_features = folds[FOLD_NAMES[0]][0].dataset.x.shape[1]
    for fold, (train_loader, val_loader, anomaly) in folds.items():
        model, run_id = experiment.fit(
            lambda: MeterMLP(n_features, batch_norm=True), train_loader, run_name=f"mlp_4out_mean_bn | {fold}", epochs=EPOCHS,
            params={"n_train": N_TRAIN, "n_val": N_VAL}, tags={"fold": fold},
        )
        pred, y = predict(model, val_loader), val_loader.dataset.y.numpy()
        scores = {"train": experiment.history(run_id, "train_rmsle")[-1],
                  "val_all": rmsle(y, pred), "val_clean": rmsle(y[~anomaly], pred[~anomaly])}
        experiment.log_metrics(run_id, {f"rmsle_{key}": value for key, value in scores.items()})

## Results

In [ ]:
runs = experiment.runs()
print(f"MLflow session {experiment.session}: {len(runs)} runs")
results = (runs.rename(columns=lambda c: c.removeprefix("tags.").removeprefix("metrics.rmsle_"))
           .set_index("fold").loc[FOLD_NAMES, ["train", "val_all", "val_clean"]])
results.insert(0, "anomaly_share", pd.Series(anomaly_share))
results["anomaly_cost"] = results.val_all - results.val_clean
display(results.round(3))

In [ ]:
ax = results[["train", "val_all", "val_clean"]].plot.bar(figsize=(7, 4), rot=0, width=0.8)
ax.set(title="mlp_4out_mean_bn: RMSLE by held-out period", xlabel="held-out months", ylabel="RMSLE")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=7)
ax.legend(["train", "validation, all rows", "validation, clean rows"], fontsize=8)
plt.tight_layout()
if FIGURE:
    ax.figure.savefig(FIGURE, dpi=150, bbox_inches="tight")
    for run_id in runs.run_id:
        experiment.client.log_artifact(run_id, FIGURE)